# Import necessary functions

In [ ]:
import zipfile
import shutil
import os
import xml.etree.ElementTree as ET
from pathlib import Path
from math import ceil

# Input the folder containing the presentation without embedded sheets

Important to use 3-5 slides in maximum in every presentation in order for the microsoft to handle the generated errors.
Create the input and output folders in the same working directory and type their name in the provision given below.

In [ ]:
input_dir = Path("input_pptx") # Input Name to be Typed
output_dir = Path("output_pptx") # Output Name to be Typed
temp_dir = Path("pptx_temp")
output_dir.mkdir(exist_ok=True)

# Python Script for Converting Presentation

In [ ]:
ns = {
    'c': "http://schemas.openxmlformats.org/drawingml/2006/chart",
    'a': "http://schemas.openxmlformats.org/drawingml/2006/main",
    'r': "http://schemas.openxmlformats.org/officeDocument/2006/relationships"
}


def on_rm_error(func, path, exc_info):
    os.chmod(path, stat.S_IWRITE)
    func(path)
    
def repackage_pptx(temp_dir: Path, output_path: Path):
    """Re-zip the temp_dir into a valid .pptx with proper internal structure."""
    with zipfile.ZipFile(output_path, 'w', zipfile.ZIP_DEFLATED) as pptx:
        for folder, _, files in os.walk(temp_dir):
            for fname in files:
                full_path = Path(folder) / fname
                # Use forward slashes inside ZIP (required for Office Open XML)
                arcname = full_path.relative_to(temp_dir).as_posix()
                pptx.write(full_path, arcname)

def col_letter(n):
    """Convert 0-based column index to Excel letter (e.g., 0 -> A, 26 -> AA)"""
    result = ''
    while n >= 0:
        result = chr(n % 26 + 65) + result
        n = n // 26 - 1
    return result

def detect_shape_and_generate_formula(pt_count, col_count, is_str=True):
    """Generate Excel formula range based on number of points and columns"""
    start_row = 2
    end_row = start_row + pt_count - 1
    start_col = 0 if is_str else 1  # A for cat, B for val
    end_col = start_col + col_count - 1

    if col_count == 1:
        cell_range = f"Sheet1!${col_letter(start_col)}${start_row}:${col_letter(end_col)}${end_row}"
    else:
        cell_range = f"Sheet1!${col_letter(start_col)}${start_row}:${col_letter(end_col)}${end_row}"

    return cell_range

def convert_lit_to_ref(parent_elem, tag_name):
    lit_tag = 'strLit' if tag_name == 'cat' else 'numLit'
    ref_tag = 'strRef' if tag_name == 'cat' else 'numRef'
    cache_tag = 'strCache' if tag_name == 'cat' else 'numCache'

    lit_elem = parent_elem.find(f'c:{tag_name}/c:{lit_tag}', ns)
    if lit_elem is None:
        return

    # Detect row count
    pt_list = lit_elem.findall('c:pt', ns)
    pt_count = len(pt_list)
    pt_count_elem = lit_elem.find('c:ptCount', ns)
    col_count = 1

    # Detect column count from multi-col structure (e.g., multiple <c:pt> with delimited <v>)
    for pt in pt_list:
        v = pt.find('c:v', ns)
        if v is not None and "," in v.text:
            col_count = len(v.text.split(","))
            break

    cell_formula = detect_shape_and_generate_formula(pt_count, col_count, is_str=(tag_name == 'cat'))

    # Build new structure
    ref_elem = ET.Element(f"{{{ns['c']}}}{ref_tag}")
    f = ET.SubElement(ref_elem, f"{{{ns['c']}}}f")
    f.text = cell_formula

    cache_elem = ET.SubElement(ref_elem, f"{{{ns['c']}}}{cache_tag}")
    if pt_count_elem is not None:
        cache_elem.append(pt_count_elem)

    for pt in pt_list:
        cache_elem.append(pt)

    old_elem = parent_elem.find(f'c:{tag_name}', ns)
    if old_elem is not None:
        parent_elem.remove(old_elem)
        new_tag = ET.Element(f"{{{ns['c']}}}{tag_name}")
        new_tag.append(ref_elem)
        parent_elem.append(new_tag)

def remove_external_data(chart_root):
    external = chart_root.find('.//c:externalData', ns)
    if external is not None:
        chart_root.remove(external)

def process_chart(chart_file):
    tree = ET.parse(chart_file)
    root = tree.getroot()
    if root.find('.//c:externalData', ns) is None:
        return

    ser = root.find('.//c:ser', ns)
    if ser is not None:
        print(f"Embedding chart in {chart_file.name}")
        convert_lit_to_ref(ser, 'cat')
        convert_lit_to_ref(ser, 'val')
        remove_external_data(root)
        tree.write(chart_file, encoding='utf-8', xml_declaration=True)

def process_pptx_file(input_path: Path, output_path: Path):
    if temp_dir.exists():
        shutil.rmtree(temp_dir, onerror=on_rm_error)
    temp_dir.mkdir()

    with zipfile.ZipFile(input_path, 'r') as zip_ref:
        zip_ref.extractall(temp_dir)

    charts_dir = temp_dir / "ppt" / "charts"
    for chart_file in charts_dir.glob("chart*.xml"):
        process_chart(chart_file)

    repackage_pptx(temp_dir, output_path)

    shutil.rmtree(temp_dir, onerror=on_rm_error)

# Batch process all pptx
for pptx_file in input_dir.glob("*.pptx"):
    output_file = output_dir / pptx_file.name
    process_pptx_file(pptx_file, output_file)

print("✅ All PPTX files processed with per-chart embedding.")